# 0 环境与参数

In [ ]:
# -*- coding: utf-8 -*-
"""
湘财证券《商品期货的多因子应用之因子筛选》(CTA系列专题三) 复现 —— 聚宽研究环境
数据: 聚宽 get_all_securities / get_price / finance.FUT_WAREHOUSE_RECEIPT
输出: 不写磁盘, 数据取来即用, 结果直接 display / print
"""
import datetime
import gc
import warnings
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from jqdata import finance

try:
    query
except NameError:
    from jqdata import *

warnings.filterwarnings('ignore')
pd.set_option('display.width', 220)
pd.set_option('display.max_rows', 200)
pd.set_option('display.unicode.east_asian_width', True)
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'WenQuanYi Micro Hei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# ---------- 研报 2.1: 因子测试以 2015年 ~ 2025年2月 为样本内 ----------
TEST_START = '2015-01-01'
TEST_END = '2025-02-28'
WARMUP_START = '2014-07-01'          # 滚动窗口预热起点
N_GROUP = 5                          # 五组分层
MIN_CROSS_SECTION = 10               # 单期截面最少品种数
IR_THRESHOLD = 0.10                  # 有效因子: |IR| 与 |Rank IR| 均不低于该值
IDIOSYNCRATIC_WINDOW = 20            # 特质波动率回归窗口(日)
WAREHOUSE_CHUNK_DAYS = 45            # 仓单分块查询天数(单次最多4000行, 超限自动二分)
PRICE_FIELDS = ['open', 'close', 'high', 'low', 'volume', 'money', 'open_interest']
LN2 = np.log(2.0)

# 调仓频率: rule为pandas周期, window为因子回看窗口, fast/slow为快慢均线窗口
FREQ_PARAMS = OrderedDict([
    ('月频', dict(rule='M', window=20, fast=5, slow=20)),
    ('周频', dict(rule='W', window=10, fast=3, slow=10)),
])

# 因子清单, 顺序与研报表7/表8一致 (共36个)
FACTOR_GROUPS = OrderedDict([
    ('动量因子', ['主力涨幅', '上涨次数占比', '涨幅占比', '均线偏离比值', '快慢均线偏离率', '隔夜趋势因子',
              '日内动量因子', '日内累计振幅因子', '日内RS波动趋势因子', '日内GK波动趋势因子',
              '日内PK波动趋势因子', '日内YZ波动趋势因子']),
    ('期限因子', ['展期收益率', '基差因子', '基差动量']),
    ('量价因子', ['偏度因子', '流动性因子', '非流动性因子', '均价突破因子', '标准化均价突破因子']),
    ('波动率因子', ['变异系数因子', '变异系数因子2', '特质波动率', 'RS波动率因子', 'GK波动率因子',
               'PK波动率因子', 'YZ波动率因子', '波动率因子']),
    ('库存因子', ['仓单水平', '仓单变化比值', '仓单涨幅']),
    ('持仓因子', ['持仓量变化比值', '持仓量涨幅', '持仓金额变化比值', '持仓金额涨幅', '对冲压力因子']),
])
ALL_FACTOR_NAMES = [name for names in FACTOR_GROUPS.values() for name in names]
FACTOR_CATEGORY = {name: category for category, names in FACTOR_GROUPS.items() for name in names}

# 研报中的五组分层图: (图号, 因子名称)
REPORT_FIGURES = OrderedDict([
    ('月频', [(1, '主力涨幅'), (2, '上涨次数占比'), (3, '涨幅占比'), (4, '均线偏离比值'), (5, '展期收益率'),
            (6, '基差因子'), (7, '流动性因子'), (8, '非流动性因子'), (9, '变异系数因子'), (10, '波动率因子'),
            (11, '仓单水平'), (12, '仓单变化比值'), (13, '持仓金额变化比值'), (14, '对冲压力因子')]),
    ('周频', [(15, '上涨次数占比'), (16, '快慢均线偏离率'), (17, '展期收益率'), (18, '基差因子'), (19, '偏度因子'),
            (20, '非流动性因子'), (21, 'GK波动率因子'), (22, '波动率因子'), (23, '仓单水平'),
            (24, '仓单变化比值'), (25, '持仓量变化比值'), (26, '对冲压力因子')]),
])
TABLE_NUMBER = {'月频': 7, '周频': 8}

### 0.1 数据获取函数

研报未涉及的数据准备部分: 合约列表、行情、仓单、交易日历。

In [ ]:
def list_commodity_contracts(window_start, window_end):
    """商品期货全部具体合约(剔除金融期货与9999/8888连续合约), 返回 code / end_date / variety"""
    securities = get_all_securities(['futures'])
    codes = pd.Series(securities.index, index=securities.index)
    is_commodity = codes.str.match(r'^[A-Za-z]{1,2}\d{3,4}\.(?:XDCE|XZCE|XSGE|XINE|GFEX)$')
    is_commodity = is_commodity & (~codes.str.contains(r'(?:9999|8888)\.'))
    securities = securities[is_commodity.values].copy()
    securities['start_date'] = pd.to_datetime(securities['start_date'])
    securities['end_date'] = pd.to_datetime(securities['end_date'])
    securities = securities[(securities['end_date'] >= window_start) & (securities['start_date'] <= window_end)]
    contracts = pd.DataFrame({'code': list(securities.index), 'end_date': securities['end_date'].values})
    contracts['variety'] = contracts['code'].str.extract(r'^([A-Za-z]+)', expand=False).str.upper()
    return contracts


def fetch_daily_prices(codes, start, end, chunk=30):
    """分批取日行情, 批量失败则逐个重试; 返回长表(time, code, 各行情字段)"""
    frames = []
    for i in range(0, len(codes), chunk):
        part = codes[i:i + chunk]
        try:
            frames.append(get_price(part, start_date=start, end_date=end, frequency='daily',
                                    fields=PRICE_FIELDS, skip_paused=False, panel=False))
        except Exception:
            for code in part:
                try:
                    one = get_price(code, start_date=start, end_date=end, frequency='daily',
                                    fields=PRICE_FIELDS, skip_paused=False).copy()
                    one['code'] = code
                    one['time'] = one.index
                    frames.append(one)
                except Exception:
                    pass
    if not frames:
        return None
    raw = pd.concat(frames, ignore_index=True)
    raw['time'] = pd.to_datetime(raw['time'])
    return raw.drop_duplicates(['time', 'code'])


def query_warehouse_range(variety, day_from, day_to, collected):
    """查询某品种一段时间的仓单明细; 触及4000行上限时按时间二分"""
    table = finance.FUT_WAREHOUSE_RECEIPT
    result = finance.run_query(query(table.day, table.underlying_code, table.warehouse_name,
                                     table.warehouse_receipt_number).filter(
        table.underlying_code == variety, table.day >= day_from, table.day <= day_to))
    if result is None or len(result) == 0:
        return
    if len(result) >= 3990 and (day_to - day_from).days >= 1:
        middle = day_from + datetime.timedelta(days=(day_to - day_from).days // 2)
        query_warehouse_range(variety, day_from, middle, collected)
        query_warehouse_range(variety, middle + datetime.timedelta(days=1), day_to, collected)
    else:
        collected.append(result)


def fetch_warehouse_receipt(variety, day_from, day_to):
    """某品种每日仓单合计(所有仓库求和), 返回 Series; 无数据返回 None"""
    collected, current = [], day_from
    while current <= day_to:
        chunk_end = min(current + datetime.timedelta(days=WAREHOUSE_CHUNK_DAYS - 1), day_to)
        try:
            query_warehouse_range(variety, current, chunk_end, collected)
        except Exception:
            pass
        current = chunk_end + datetime.timedelta(days=1)
    if not collected:
        return None
    detail = pd.concat(collected, ignore_index=True)
    detail['day'] = pd.to_datetime(detail['day'])
    detail['warehouse_receipt_number'] = detail['warehouse_receipt_number'].astype(float)
    detail = detail.drop_duplicates(['day', 'warehouse_name'])
    return detail.groupby('day')['warehouse_receipt_number'].sum()


def get_trade_calendar(start, end):
    """交易日历: 优先 get_trade_days, 失败则用螺纹钢连续合约日期, 最后退回工作日"""
    start_text, end_text = start.strftime('%Y-%m-%d'), end.strftime('%Y-%m-%d')
    try:
        return pd.DatetimeIndex(pd.to_datetime(list(get_trade_days(start_date=start_text, end_date=end_text))))
    except Exception:
        pass
    try:
        prices = get_price('RB9999.XSGE', start_date=start_text, end_date=end_text,
                           frequency='daily', fields=['close'], skip_paused=False)
        return pd.DatetimeIndex(pd.to_datetime(prices.index))
    except Exception:
        return pd.bdate_range(start, end)

### 0.2 品种级序列构建函数

主力合约: 持仓量最大且只向更远月切换; 近月/远月: 持仓量最大的两个合约按到期日排序。收益用"前一日持有的主力合约"当日涨跌计算, 无换月跳空、无未来函数。

In [ ]:
def pick_by_column(matrix, column_index, lag=0):
    """result[i] = matrix[i - lag, column_index[i]]; column_index<0 表示当日无合约"""
    result = np.full(len(column_index), np.nan)
    rows = np.arange(len(column_index)) - lag
    valid = (column_index >= 0) & (rows >= 0)
    result[valid] = matrix[rows[valid], column_index[valid]]
    return result


def select_main_near_far(close, volume, open_interest, expiry, dates):
    """逐日选出主力/近月/远月合约在矩阵中的列位置(-1表示无)"""
    length = len(dates)
    main = np.full(length, -1, dtype=int)
    near = np.full(length, -1, dtype=int)
    far = np.full(length, -1, dtype=int)
    current = -1
    with np.errstate(invalid='ignore'):
        for i in range(length):
            alive = (volume[i] > 0) & (open_interest[i] > 0) & (expiry >= dates.values[i]) & (~np.isnan(close[i]))
            if not alive.any():
                current = -1
                continue
            alive_open_interest = np.where(alive, open_interest[i], -1.0)
            candidate = int(np.argmax(alive_open_interest))
            if current == -1 or (not alive[current]) or (candidate != current and expiry[candidate] > expiry[current]):
                current = candidate                                   # 只允许向更远月切换
            main[i] = current
            if alive.sum() >= 2:
                first, second = np.argsort(-alive_open_interest)[:2]
                if expiry[first] > expiry[second]:
                    first, second = second, first
                near[i], far[i] = first, second
    return main, near, far


def build_variety_data(contract_codes, expiry_by_code, price_panels, dates):
    """把一个品种的全部合约行情整理成: 主力日行情 / 连续价格指数 / 全合约持仓 / 近远月价格"""
    arrays = {field: price_panels[field][contract_codes].values for field in PRICE_FIELDS}
    expiry = pd.DatetimeIndex([expiry_by_code[code] for code in contract_codes]).values
    main, near, far = select_main_near_far(arrays['close'], arrays['volume'], arrays['open_interest'], expiry, dates)
    held = np.concatenate([[-1], main[:-1]])                          # 前一日选出的主力, 当日持有

    daily = pd.DataFrame(index=dates)
    daily['open'] = pick_by_column(arrays['open'], held)
    daily['high'] = pick_by_column(arrays['high'], held)
    daily['low'] = pick_by_column(arrays['low'], held)
    daily['close'] = pick_by_column(arrays['close'], held)
    daily['prev_close'] = pick_by_column(arrays['close'], held, 1)
    daily['prev_open'] = pick_by_column(arrays['open'], held, 1)
    daily['volume'] = pick_by_column(arrays['volume'], held)
    daily['amount'] = pick_by_column(arrays['money'], held)
    daily['ret'] = daily['close'] / daily['prev_close'] - 1
    daily.loc[daily['ret'].abs() > 0.5, 'ret'] = np.nan               # 剔除异常涨跌幅
    first_valid = daily['ret'].first_valid_index()
    if first_valid is None:
        return None
    filled_ret = daily['ret'].copy()
    filled_ret.loc[first_valid:] = filled_ret.loc[first_valid:].fillna(0.0)
    price_index = (1 + filled_ret).cumprod()                          # 连续价格指数(换月无跳空)

    with np.errstate(invalid='ignore'):
        total_open_interest = pd.Series(np.nansum(arrays['open_interest'], axis=1), index=dates).replace(0, np.nan)
        total_amount = pd.Series(np.nansum(arrays['open_interest'] * arrays['close'], axis=1),
                                 index=dates).replace(0, np.nan)      # 持仓金额 = 持仓量 * 收盘价

    has_pair = near >= 0
    near_days = np.full(len(dates), np.nan)
    far_days = np.full(len(dates), np.nan)
    near_days[has_pair] = (expiry[near[has_pair]] - dates.values[has_pair]) / np.timedelta64(1, 'D')
    far_days[has_pair] = (expiry[far[has_pair]] - dates.values[has_pair]) / np.timedelta64(1, 'D')
    term = pd.DataFrame({'near_price': pick_by_column(arrays['close'], near),
                         'far_price': pick_by_column(arrays['close'], far),
                         'near_days': near_days, 'far_days': far_days}, index=dates)

    return dict(daily=daily, price_index=price_index, total_open_interest=total_open_interest,
                total_amount=total_amount, term=term, held=held, near=near, far=far,
                close_matrix=arrays['close'].astype('float32'),
                open_interest_matrix=arrays['open_interest'].astype('float32'))


def load_variety_data(contracts, expiry_by_code, calendar, start, end):
    """逐品种: 取行情 -> 构建序列 -> 取仓单; 逐个释放原始数据以节省内存"""
    variety_data_by_name, warehouse_by_name = OrderedDict(), OrderedDict()
    varieties = sorted(contracts['variety'].unique())
    for position, variety in enumerate(varieties, 1):
        codes = list(contracts.loc[contracts['variety'] == variety, 'code'])
        raw = fetch_daily_prices(codes, start.strftime('%Y-%m-%d'), end.strftime('%Y-%m-%d'))
        if raw is None:
            continue
        price_panels = {field: raw.pivot(index='time', columns='code', values=field).reindex(calendar)
                        for field in PRICE_FIELDS}
        del raw
        available = [code for code in codes if code in price_panels['close'].columns]
        variety_data = build_variety_data(available, expiry_by_code, price_panels, calendar) if available else None
        del price_panels
        if variety_data is not None and variety_data['daily']['close'].notna().sum() >= 60:
            variety_data_by_name[variety] = variety_data
            warehouse = fetch_warehouse_receipt(variety, start.date(), end.date())
            if warehouse is not None:
                warehouse_by_name[variety] = warehouse.reindex(calendar).ffill(limit=5)
        gc.collect()
        if position % 10 == 0 or position == len(varieties):
            print('品种进度 %d/%d' % (position, len(varieties)))
    return variety_data_by_name, warehouse_by_name

### 0.3 读取数据

In [ ]:
test_start = pd.Timestamp(TEST_START)
test_end = min(pd.Timestamp(TEST_END), pd.Timestamp(datetime.date.today()))
warmup_start = pd.Timestamp(WARMUP_START)

contracts = list_commodity_contracts(warmup_start, test_end)
calendar = get_trade_calendar(warmup_start, test_end)
expiry_by_code = dict(zip(contracts['code'], contracts['end_date']))
variety_data_by_name, warehouse_by_name = load_variety_data(contracts, expiry_by_code, calendar, warmup_start, test_end)

# 品种数据概览
variety_summary = pd.DataFrame({
    '合约数': contracts.groupby('variety').size(),
    '有效交易日数': pd.Series(OrderedDict((name, int(data['daily']['close'].notna().sum()))
                                    for name, data in variety_data_by_name.items())),
    '有仓单数据': pd.Series(OrderedDict((name, name in warehouse_by_name) for name in variety_data_by_name)),
}).dropna(subset=['有效交易日数'])
print('测试区间: %s ~ %s, 可用品种数: %d' % (test_start.date(), test_end.date(), len(variety_data_by_name)))
display(variety_summary)

# 品种 x 日期 面板: 日收益 / 连续价格指数 / 是否有交易
return_panel = pd.DataFrame(OrderedDict((name, data['daily']['ret']) for name, data in variety_data_by_name.items()))
price_index_panel = pd.DataFrame(OrderedDict((name, data['price_index']) for name, data in variety_data_by_name.items()))
tradable_panel = pd.DataFrame(OrderedDict((name, data['daily']['close'].notna()) for name, data in variety_data_by_name.items()))

# 1 商品期货的因子体系构建

研报共构建 36 个因子, 分六类: 动量12 / 期限3 / 量价5 / 持仓5 / 库存3 / 波动率8。各类因子函数统一输入"单个品种数据 + 频率参数", 输出该品种的因子时间序列 DataFrame。

## 1.1 动量因子

价格动量(主力涨幅、上涨次数占比、涨幅占比)、形态动量(均线偏离比值、快慢均线偏离率)、微观结构动量(隔夜、日内、日内振幅及 RS/GK/PK/YZ 波动趋势)。

In [ ]:
def minimum_observations(window):
    """滚动窗口最少有效观测数"""
    return max(3, int(window * 0.7))


def yang_zhang_weight(window):
    """Yang-Zhang 波动率中收盘价方差的权重 ω = 0.34 / (1.34 + (N+1)/(N-1))"""
    return 0.34 / (1.34 + (window + 1.0) / (window - 1.0))


def intraday_volatility_estimates(daily, window):
    """单日 RS / GK / PK / YZ 波动率估计(未开方), 以及开盘、收盘对数收益; 动量与波动率因子共用"""
    log_high_low = np.log(daily['high'] / daily['low'])
    log_close_open = np.log(daily['close'] / daily['open'])
    estimates = pd.DataFrame(index=daily.index)
    estimates['RS'] = (np.log(daily['high'] / daily['close']) * np.log(daily['high'] / daily['open'])
                       + np.log(daily['low'] / daily['close']) * np.log(daily['low'] / daily['open']))
    estimates['GK'] = 0.5 * log_high_low ** 2 - (2 * LN2 - 1) * log_close_open ** 2
    estimates['PK'] = log_high_low ** 2 / (4 * LN2)
    estimates['open_return'] = np.log(daily['open'] / daily['prev_open'])      # ln(o_t / o_{t-1})
    estimates['close_return'] = np.log(daily['close'] / daily['prev_close'])   # ln(c_t / c_{t-1})
    omega = yang_zhang_weight(window)
    estimates['YZ'] = estimates['open_return'] ** 2 + omega * estimates['close_return'] ** 2 + (1 - omega) * estimates['RS']
    return estimates


def compute_momentum_factors(variety_data, freq_params):
    """1.1 动量因子 (表1), 共12个"""
    window, fast, slow = freq_params['window'], freq_params['fast'], freq_params['slow']
    min_obs = minimum_observations(window)
    daily, price_index = variety_data['daily'], variety_data['price_index']
    ret = daily['ret']
    rolling = lambda series: series.rolling(window, min_periods=min_obs)
    close_open_sign = np.sign(daily['close'] - daily['open'])
    estimates = intraday_volatility_estimates(daily, window)

    factors = OrderedDict()
    # 价格动量
    factors['主力涨幅'] = price_index / price_index.shift(window - 1) - 1
    factors['上涨次数占比'] = rolling((ret > 0).astype(float).where(ret.notna())).mean()
    factors['涨幅占比'] = rolling(ret.clip(lower=0)).sum() / rolling(ret.abs()).sum()
    # 形态动量
    factors['均线偏离比值'] = -rolling(price_index).mean() / price_index
    fast_average, slow_average = price_index.rolling(fast).mean(), price_index.rolling(slow).mean()
    factors['快慢均线偏离率'] = (fast_average - slow_average) / slow_average
    # 微观结构动量
    factors['隔夜趋势因子'] = rolling(daily['open'] / daily['prev_close'] - 1).mean()
    factors['日内动量因子'] = rolling(daily['close'] / daily['open'] - 1).mean()
    amplitude = (2 * (daily['high'] - daily['low']) * close_open_sign - (daily['close'] - daily['open'])) / daily['close']
    factors['日内累计振幅因子'] = rolling(amplitude).mean()
    for label in ['RS', 'GK', 'PK', 'YZ']:
        factors['日内%s波动趋势因子' % label] = rolling(close_open_sign * np.sqrt(estimates[label].clip(lower=0))).mean()
    return pd.DataFrame(factors)

## 1.2 期限结构因子

展期收益率 = (ln P1 - ln P2) * 365 / (N2 - N1) (研报公式 ln(P1-P2) 应为 lnP1-lnP2 的笔误); 基差因子 = (P1/P2 - 1) * 365 / (N2 - N1); 基差动量 = 近月与远月合约最近N日累计收益之差。

In [ ]:
def contract_cumulative_return(variety_data, column_index, window):
    """近月(或远月)合约最近 window 日累计收益"""
    matrix = variety_data['close_matrix']
    values = pick_by_column(matrix, column_index) / pick_by_column(matrix, column_index, window) - 1
    return pd.Series(values, index=variety_data['daily'].index)


def compute_term_structure_factors(variety_data, freq_params):
    """1.2 期限结构因子 (表2), 共3个"""
    term = variety_data['term']
    days_gap = term['far_days'] - term['near_days']
    days_gap = days_gap.where(days_gap > 0)

    factors = OrderedDict()
    factors['展期收益率'] = (np.log(term['near_price']) - np.log(term['far_price'])) * 365.0 / days_gap
    factors['基差因子'] = (term['near_price'] / term['far_price'] - 1) * 365.0 / days_gap
    factors['基差动量'] = (contract_cumulative_return(variety_data, variety_data['near'], freq_params['window'])
                       - contract_cumulative_return(variety_data, variety_data['far'], freq_params['window']))
    return pd.DataFrame(factors)

## 1.3 量价因子

偏度因子捕捉收益率分布的不对称性; 流动性/非流动性因子从不同角度度量交易成本与市场深度; 均价突破因子及其标准化版本刻画价格偏离近期均价的强度。

In [ ]:
def compute_price_volume_factors(variety_data, freq_params):
    """1.3 量价因子 (表3), 共5个"""
    window = freq_params['window']
    min_obs = minimum_observations(window)
    daily, price_index = variety_data['daily'], variety_data['price_index']
    ret = daily['ret']
    rolling = lambda series: series.rolling(window, min_periods=min_obs)
    moving_average = rolling(price_index).mean()

    factors = OrderedDict()
    factors['偏度因子'] = rolling(ret).skew()
    factors['流动性因子'] = rolling(daily['volume'] / ret.abs().replace(0, np.nan)).mean()      # 成交量 / |收益率|
    factors['非流动性因子'] = rolling(ret.abs() / daily['amount'].replace(0, np.nan)).mean()    # |收益率| / 成交金额
    factors['均价突破因子'] = price_index - moving_average
    factors['标准化均价突破因子'] = (price_index - moving_average) / (rolling(ret).std() * price_index.shift(window - 1))
    return pd.DataFrame(factors)

## 1.4 持仓因子

持仓量/持仓金额变化使用品种全部合约合计; 对冲压力因子按研报使用主力合约: 同一合约 N 日持仓量变化 / N 日成交量合计(避免换月造成持仓跳变)。

In [ ]:
def compute_open_interest_factors(variety_data, freq_params):
    """1.4 持仓因子 (表4), 共5个"""
    window = freq_params['window']
    min_obs = minimum_observations(window)
    daily = variety_data['daily']
    total_open_interest, total_amount = variety_data['total_open_interest'], variety_data['total_amount']

    # 主力合约(当日持有的合约)的当前持仓量 与 N日前持仓量
    held, open_interest_matrix = variety_data['held'], variety_data['open_interest_matrix']
    main_open_interest_change = pd.Series(pick_by_column(open_interest_matrix, held)
                                          - pick_by_column(open_interest_matrix, held, window), index=daily.index)

    factors = OrderedDict()
    factors['持仓量变化比值'] = total_open_interest / total_open_interest.shift(window)
    factors['持仓量涨幅'] = total_open_interest / total_open_interest.shift(window) - 1
    factors['持仓金额变化比值'] = total_amount / total_amount.shift(window)
    factors['持仓金额涨幅'] = total_amount / total_amount.shift(window) - 1
    factors['对冲压力因子'] = main_open_interest_change / daily['volume'].rolling(window, min_periods=min_obs).sum()
    return pd.DataFrame(factors)

## 1.5 库存因子

以交易所仓单为库存代理: 仓单水平 = 当前仓单 / 前N日仓单均值; 仓单变化比值、仓单涨幅刻画仓单自身变动趋势。

In [ ]:
def compute_inventory_factors(variety_data, freq_params, warehouse):
    """1.5 库存因子 (表5), 共3个; warehouse 为该品种每日仓单合计(缺失则全为NaN)"""
    window = freq_params['window']
    min_obs = minimum_observations(window)
    if warehouse is None:
        warehouse = pd.Series(np.nan, index=variety_data['daily'].index)

    factors = OrderedDict()
    factors['仓单水平'] = warehouse / warehouse.shift(1).rolling(window, min_periods=min_obs).mean()
    factors['仓单变化比值'] = warehouse / warehouse.shift(window)
    factors['仓单涨幅'] = warehouse / warehouse.shift(window) - 1
    return pd.DataFrame(factors)

## 1.6 波动率因子

三类: 变异系数(风险收益比)、特质波动率(剔除市场/动量/基差因子后的残差波动)、历史波动率(RS/GK/PK/YZ 与简单标准差)。特质波动率需要全品种截面信息, 在 2.1 统一计算后回填。

In [ ]:
def compute_volatility_factors(variety_data, freq_params):
    """1.6 波动率因子 (表6), 共8个; 特质波动率先占位, 由 fill_idiosyncratic_volatility 回填"""
    window = freq_params['window']
    min_obs = minimum_observations(window)
    daily = variety_data['daily']
    ret = daily['ret']
    rolling = lambda series: series.rolling(window, min_periods=min_obs)
    estimates = intraday_volatility_estimates(daily, window)
    mean_ret, std_ret = rolling(ret).mean(), rolling(ret).std()
    omega = yang_zhang_weight(window)

    factors = OrderedDict()
    factors['变异系数因子'] = std_ret ** 2 / mean_ret           # 方差 / 均值
    factors['变异系数因子2'] = std_ret / mean_ret               # 标准差 / 均值
    factors['特质波动率'] = pd.Series(np.nan, index=daily.index)
    for label in ['RS', 'GK', 'PK']:
        factors['%s波动率因子' % label] = np.sqrt(rolling(estimates[label]).mean().clip(lower=0))
    yang_zhang = (rolling(estimates['open_return']).var() + omega * rolling(estimates['close_return']).var()
                  + (1 - omega) * rolling(estimates['RS']).mean())
    factors['YZ波动率因子'] = np.sqrt(yang_zhang.clip(lower=0))
    factors['波动率因子'] = std_ret                              # 收益率标准差
    return pd.DataFrame(factors)


def long_short_factor_return(factor_panel, return_panel, quantile=0.3):
    """用前一日因子排序, 构造当日 多头(前30%) - 空头(后30%) 的因子收益"""
    rank = factor_panel.shift(1).rank(axis=1, pct=True)
    long_return = return_panel.where(rank >= 1 - quantile).mean(axis=1)
    short_return = return_panel.where(rank <= quantile).mean(axis=1)
    return (long_return - short_return).fillna(0.0)


def rolling_idiosyncratic_volatility(return_panel, market, momentum, basis, window):
    """各品种收益对 市场/动量/基差 因子收益滚动回归, 取残差标准差"""
    regressors = np.column_stack([market.values, momentum.values, basis.values, np.ones(len(return_panel))])
    required = max(8, int(window * 0.7))
    result = pd.DataFrame(np.nan, index=return_panel.index, columns=return_panel.columns)
    for variety in return_panel.columns:
        target = return_panel[variety].values
        residual_std = np.full(len(target), np.nan)
        for i in range(window - 1, len(target)):
            y = target[i - window + 1:i + 1]
            x = regressors[i - window + 1:i + 1]
            valid = (~np.isnan(y)) & (~np.isnan(x).any(axis=1))
            if valid.sum() < required:
                continue
            beta = np.linalg.lstsq(x[valid], y[valid], rcond=None)[0]
            residual_std[i] = np.std(y[valid] - x[valid].dot(beta), ddof=4)
        result[variety] = residual_std
    return result


def fill_idiosyncratic_volatility(factor_panels_by_freq, return_panel):
    """用首个频率的 主力涨幅 / 展期收益率 构造动量、基差因子收益, 回填各频率的特质波动率"""
    first_panels = list(factor_panels_by_freq.values())[0]
    market = return_panel.mean(axis=1).fillna(0.0)
    momentum = long_short_factor_return(first_panels['主力涨幅'], return_panel)
    basis = long_short_factor_return(first_panels['展期收益率'], return_panel)
    idiosyncratic = rolling_idiosyncratic_volatility(return_panel, market, momentum, basis, IDIOSYNCRATIC_WINDOW)
    for panels in factor_panels_by_freq.values():
        panels['特质波动率'] = idiosyncratic

# 2 有效因子的筛选

## 2.1 因子的构建与调仓频率

窗口期随调仓频率调整(月频 N=20, 周频 N=10)。因子评价从 ICIR 与五组分层两方面判断: IC/Rank IC 取调仓日因子与下一调仓期收益的截面相关; IR = IC均值 / IC标准差; 五组分层 group1 为因子值最高组、group5 为最低组; 多空收益 = 样本期 group1 与 group5 累计净值之差 (IC均值为负的因子反向), 对应研报表7/表8。

In [ ]:
def compute_all_factors(variety_data, freq_params, warehouse):
    """单个品种的 36 个因子, 列顺序与研报表7一致"""
    blocks = [compute_momentum_factors(variety_data, freq_params),
              compute_term_structure_factors(variety_data, freq_params),
              compute_price_volume_factors(variety_data, freq_params),
              compute_open_interest_factors(variety_data, freq_params),
              compute_inventory_factors(variety_data, freq_params, warehouse),
              compute_volatility_factors(variety_data, freq_params)]
    return pd.concat(blocks, axis=1).replace([np.inf, -np.inf], np.nan)[ALL_FACTOR_NAMES]


def build_factor_panels(variety_data_by_name, warehouse_by_name, freq_params):
    """{因子名: DataFrame(日期 x 品种)}"""
    per_variety = OrderedDict((name, compute_all_factors(data, freq_params, warehouse_by_name.get(name)))
                              for name, data in variety_data_by_name.items())
    return OrderedDict((factor, pd.DataFrame(OrderedDict((name, per_variety[name][factor]) for name in per_variety)))
                       for factor in ALL_FACTOR_NAMES)


def winsorize_cross_section(factor_panel, multiple=3.0):
    """每个截面按 中位数 ± 3倍MAD 去极值"""
    median = factor_panel.median(axis=1)
    mad = factor_panel.sub(median, axis=0).abs().median(axis=1) * 1.4826
    mad = mad.replace(0, np.nan)
    return factor_panel.clip(lower=median - multiple * mad, upper=median + multiple * mad, axis=0)


def get_rebalance_dates(calendar, rule, start, end):
    """每个周期(月/周)的最后一个交易日; 最后一个未走完的周期丢弃"""
    days = calendar[(calendar >= start) & (calendar <= end)]
    last_days = pd.Series(days, index=days).groupby(days.to_period(rule)).max()
    rebalance_dates = pd.DatetimeIndex(sorted(last_days.values))
    if (rebalance_dates[-1].to_period(rule).end_time.normalize() - rebalance_dates[-1]).days > 3:
        rebalance_dates = rebalance_dates[:-1]
    return rebalance_dates


def evaluate_factor(factor_panel, forward_return, rebalance_dates):
    """逐调仓期计算 IC / Rank IC 与五组平均收益"""
    factor_at_rebalance = factor_panel.reindex(rebalance_dates)
    ic_rows, group_rows, used_dates = [], [], []
    for date in rebalance_dates[:-1]:
        factor, future = factor_at_rebalance.loc[date], forward_return.loc[date]
        valid = factor.notna() & future.notna()
        if valid.sum() < MIN_CROSS_SECTION:
            continue
        factor, future = factor[valid], future[valid]
        ic = np.corrcoef(factor.values, future.values)[0, 1]
        rank_ic = np.corrcoef(factor.rank().values, future.rank().values)[0, 1]
        group_label = pd.qcut(factor.rank(method='first', ascending=False), N_GROUP, labels=False)
        group_return = future.groupby(group_label.values).mean().reindex(range(N_GROUP))
        ic_rows.append([ic, rank_ic])
        group_rows.append(group_return.values)
        used_dates.append(date)
    if not used_dates:
        return None
    ic_table = pd.DataFrame(ic_rows, index=used_dates, columns=['IC', 'Rank IC'])
    group_returns = pd.DataFrame(group_rows, index=used_dates, columns=['group%d' % (i + 1) for i in range(N_GROUP)])
    return ic_table, group_returns


def summarize_factor(ic_table, group_returns):
    """IR / Rank IR / IC / Rank IC / 多空收益(累计净值差, IC为负则反向)"""
    ic_mean, rank_ic_mean = ic_table['IC'].mean(), ic_table['Rank IC'].mean()
    direction = 1.0 if ic_mean >= 0 else -1.0
    final_net_value = (1 + group_returns.fillna(0)).cumprod().iloc[-1]
    return OrderedDict([('IR', ic_mean / ic_table['IC'].std()), ('Rank IR', rank_ic_mean / ic_table['Rank IC'].std()),
                        ('IC', ic_mean), ('Rank IC', rank_ic_mean),
                        ('多空收益', direction * (final_net_value['group1'] - final_net_value['group%d' % N_GROUP]))])


def test_frequency(freq_name, factor_panels, price_index_panel, tradable_panel, calendar, test_start, test_end):
    """某调仓频率下全部因子的有效性测试: 返回结果表(研报表7/表8)与各因子五组收益"""
    rebalance_dates = get_rebalance_dates(calendar, FREQ_PARAMS[freq_name]['rule'], test_start, test_end)
    price_at_rebalance = price_index_panel.reindex(rebalance_dates)
    forward_return = price_at_rebalance.shift(-1) / price_at_rebalance - 1
    tradable_now = tradable_panel.reindex(rebalance_dates)
    tradable_next = tradable_now.astype(float).shift(-1).fillna(0).astype(bool)
    forward_return = forward_return.where(tradable_now & tradable_next)
    print('%s调仓: 窗口N=%d, 调仓期数=%d (%s ~ %s)' % (freq_name, FREQ_PARAMS[freq_name]['window'],
                                               len(rebalance_dates) - 1, rebalance_dates[0].date(), rebalance_dates[-1].date()))

    rows, group_returns_by_factor = [], OrderedDict()
    for category, names in FACTOR_GROUPS.items():
        for name in names:
            evaluation = evaluate_factor(winsorize_cross_section(factor_panels[name]), forward_return, rebalance_dates)
            if evaluation is None:
                rows.append(OrderedDict([('因子类型', category), ('因子名称', name)]))
                continue
            group_returns_by_factor[name] = evaluation[1]
            rows.append(OrderedDict([('因子类型', category), ('因子名称', name)] + list(summarize_factor(*evaluation).items())))
    table = pd.DataFrame(rows, columns=['因子类型', '因子名称', 'IR', 'Rank IR', 'IC', 'Rank IC', '多空收益'])
    table['有效'] = np.where((table['IR'].abs() >= IR_THRESHOLD) & (table['Rank IR'].abs() >= IR_THRESHOLD), '★', '')
    return table, group_returns_by_factor

### 2.1.1 因子面板计算

In [ ]:
factor_panels_by_freq = OrderedDict((freq_name, build_factor_panels(variety_data_by_name, warehouse_by_name, freq_params))
                                    for freq_name, freq_params in FREQ_PARAMS.items())
fill_idiosyncratic_volatility(factor_panels_by_freq, return_panel)

### 2.1.2 结果展示函数

In [ ]:
def format_result_table(table):
    """保留两位小数, 多空收益显示为百分比"""
    shown = table.reset_index(drop=True).copy()
    shown[['IR', 'Rank IR', 'IC', 'Rank IC']] = shown[['IR', 'Rank IR', 'IC', 'Rank IC']].round(2)
    shown['多空收益'] = (shown['多空收益'] * 100).round(1).astype(str) + '%'
    return shown


def plot_group_net_values(freq_name, category, group_returns_by_factor):
    """画出研报中属于该类因子的五组分层净值图"""
    figures = [(number, name) for number, name in REPORT_FIGURES[freq_name]
               if FACTOR_CATEGORY[name] == category and name in group_returns_by_factor]
    if not figures:
        return
    n_cols = 2
    n_rows = int(np.ceil(len(figures) / float(n_cols)))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 3.6 * n_rows))
    axes = np.array(axes).reshape(-1)
    for ax, (number, name) in zip(axes, figures):
        group_returns = group_returns_by_factor[name]
        net_value = (1 + group_returns.fillna(0)).cumprod()
        start_row = pd.DataFrame([[1.0] * N_GROUP], columns=net_value.columns,
                                 index=[group_returns.index[0] - pd.Timedelta(days=1)])
        pd.concat([start_row, net_value]).plot(ax=ax, linewidth=1.3)
        ax.set_title('图%d %s五组分层' % (number, name))
        ax.grid(alpha=0.3)
    for ax in axes[len(figures):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()


def show_frequency_results(freq_name, table, group_returns_by_factor):
    """按研报小节顺序展示: 各类因子结果与分层图 -> 完整测试结果表 -> 有效因子"""
    for category in FACTOR_GROUPS:
        print('\n【%s %s】' % (freq_name, category))
        display(format_result_table(table[table['因子类型'] == category]))
        plot_group_net_values(freq_name, category, group_returns_by_factor)
    print('\n表%d %s因子的有效性测试结果 (多空收益为样本期累计, IC为负的因子已反向)' % (TABLE_NUMBER[freq_name], freq_name))
    display(format_result_table(table))
    effective = table.loc[table['有效'] == '★', '因子名称']
    print('%s有效因子(|IR|与|Rank IR|均>=%.2f): %s' % (freq_name, IR_THRESHOLD, '、'.join(effective) if len(effective) else '无'))
    display(table.groupby('因子类型', sort=False)['IR'].apply(lambda x: x.abs().mean()).round(3).to_frame('|IR|均值'))

## 2.2 月频调仓下的有效因子筛选

研报结论: 展期收益率与基差因子最稳健(IR、Rank IR 超过0.2, 多空收益超过12%); 流动性因子为明确有效的正向因子; 仓单水平为强反向因子; 动量因子中主力涨幅、涨幅占比相对稳健; 波动率因子与持仓因子整体无效。(2.2.1 动量 / 2.2.2 期限 / 2.2.3 量价 / 2.2.4 波动率 / 2.2.5 库存 / 2.2.6 持仓, 由 show_frequency_results 按顺序展示)

In [ ]:
monthly_table, monthly_group_returns = test_frequency('月频', factor_panels_by_freq['月频'], price_index_panel, tradable_panel, calendar,
                                                       test_start, test_end)
show_frequency_results('月频', monthly_table, monthly_group_returns)

## 2.3 周频调仓下的有效因子筛选

研报结论: 动量因子全部失效; 期限因子依然有效但弱于月频; 偏度因子为负向、GK/RS/PK 日内波动率因子转为有效; 库存因子为强负向且收益远超月频; 持仓因子仍无效。(2.3.1 动量 / 2.3.2 期限 / 2.3.3 量价 / 2.3.4 波动率 / 2.3.5 库存 / 2.3.6 持仓)

In [ ]:
weekly_table, weekly_group_returns = test_frequency('周频', factor_panels_by_freq['周频'], price_index_panel, tradable_panel, calendar,
                                                       test_start, test_end)
show_frequency_results('周频', weekly_table, weekly_group_returns)

# 3 总结

月频 vs 周频 对照: 期限因子穿越周期; 库存因子周频更强; 动量因子存在频率断层; 持仓因子普遍失效。

In [ ]:
tables_by_freq = OrderedDict([('月频', monthly_table), ('周频', weekly_table)])

# 月频 vs 周频: IR 与多空收益对照
summary = pd.concat([table.set_index('因子名称')[['IR', '多空收益']].add_suffix('_' + freq_name)
                     for freq_name, table in tables_by_freq.items()], axis=1)
summary.insert(0, '因子类型', [FACTOR_CATEGORY[name] for name in summary.index])
for column in summary.columns:
    if column.startswith('IR'):
        summary[column] = summary[column].round(2)
    elif column.startswith('多空收益'):
        summary[column] = (summary[column] * 100).round(1).astype(str) + '%'
display(summary)

# 各类因子有效因子个数
effective_count = pd.DataFrame(OrderedDict((freq_name, table.groupby('因子类型', sort=False)['有效'].apply(lambda x: int((x == '★').sum())))
                                           for freq_name, table in tables_by_freq.items()))
display(effective_count)

# 4 风险提示

市场波动风险和因子失效风险。